In [ ]:
import json
import pandas as pd

# The TRT exams of the bench, with the paths of the exam and of the answer key and the position (it chooses
# the block of the answer key)
with open('extraction_config/trt_exams.json', encoding='utf-8') as f:
    df_exams = pd.DataFrame(json.load(f))
exams_list = df_exams['exam'].tolist()
print(f'{len(df_exams)} exams')

In [ ]:
%load_ext autoreload
%autoreload 2

import json
import os
import pdfplumber
from utils.utils_trt import find_columns, fill_lacunas, split_questions, exam_type, extract_answer_key
from utils.utils_oab import extract_by_columns, join_texts, load_rules, clean_text, extract_questions

idx = 0
exam_path = df_exams.loc[idx, 'prova_path']
answer_key_path = df_exams.loc[idx, 'gabarito_path']
print(f'Prova: {exams_list[idx]}')
literals, patterns = load_rules('extraction_config/cleaning_rules_trt.json')

# Column split of each page: 1.0 is a page in a single column (whole width)
dict_mid = find_columns(exam_path)
print('Pages:', ''.join('F' if mid == 1.0 else '2' for mid in dict_mid.values()), '(F = whole width, 2 = two columns)')

In [ ]:
aa_left, aa_right = extract_by_columns(exam_path, dict_mid)

In [ ]:
questions = join_texts(aa_left, aa_right)

In [ ]:
questions = clean_text(questions, literals, patterns)
# Labels of the blanks (I, II, ...) of the fill-in-the-blank questions in their places
questions = fill_lacunas(questions)

In [ ]:
# Only the questions of Direito do Trabalho and Direito Processual do Trabalho
with open('extraction_config/trt_labor_law_question_ranges.json', encoding='utf-8') as f:
    labor_ranges = json.load(f)[exams_list[idx]]
questions_split = {
    number: question for number, question in split_questions(questions).items()
    if any(first <= number <= last for first, last in labor_ranges)
}
print(f'Labor law ranges {labor_ranges}: {len(questions_split)} questions')

In [ ]:
question_literals, question_patterns = load_rules('extraction_config/cleaning_rules_trt.json', stage='question')
extracted_questions = extract_questions(questions_split, question_literals, question_patterns, n_options=5)

In [ ]:
gabarito_text = []
with pdfplumber.open(answer_key_path) as pdf:
    for pagina in pdf.pages:
        gabarito_text.append(pagina.extract_text())

In [ ]:
# The position code and the type printed on the exam choose the block of the answer key
code, exam_type_number = exam_type(exam_path)
print(f'Position code: {code!r}, type: {exam_type_number}')
answer_key = extract_answer_key(answer_key_path, code, exam_type_number, df_exams.loc[idx, 'position'])
if not answer_key:
    print('Answer key without text (scanned image) or without the block of this exam.')
missing = [q['question_number'] for q in extracted_questions if q['question_number'] not in answer_key]
if answer_key and missing:
    print(f'Questions without answer in the answer key: {missing}')

In [ ]:
# Annulled questions ('*' or 'T' in the answer key) are not saved
annulled = [number for number, answer in answer_key.items() if answer is None and number in questions_split]
extracted_questions = [q for q in extracted_questions if q['question_number'] not in annulled]
print(f'Annulled questions removed: {len(annulled)} {annulled}')

# answer is None when the answer key has no text
for question_data in extracted_questions:
    question_data['answer'] = answer_key.get(question_data['question_number'])

In [ ]:
csv_path = 'csvs/trt/trt_questions_all_bench.csv'
os.makedirs(os.path.dirname(csv_path), exist_ok=True)
df_exam = pd.DataFrame([{
    'question_number': question_data['question_number'],
    'question': question_data['question'],
    'options': list(question_data['options'].values()),
    'exam': exams_list[idx],
    'answer': question_data['answer'],
} for question_data in extracted_questions])

if os.path.exists(csv_path):
    df_saved = pd.read_csv(csv_path, encoding='utf-8-sig')
    df_saved = df_saved[df_saved['exam'] != exams_list[idx]]
    df_exam = pd.concat([df_saved, df_exam], ignore_index=True)
df_exam.to_csv(csv_path, index=False, encoding='utf-8-sig')
print(f'{csv_path}: {len(df_exam)} questions from {df_exam["exam"].nunique()} exams')

# Annulled questions of each exam, to count them over all the exams
annulled_path = 'csvs/trt/trt_annulled_questions.csv'
df_annulled = pd.DataFrame({'exam': exams_list[idx], 'question_number': annulled})
if os.path.exists(annulled_path):
    df_saved = pd.read_csv(annulled_path, encoding='utf-8-sig')
    df_saved = df_saved[df_saved['exam'] != exams_list[idx]]
    df_annulled = pd.concat([df_saved, df_annulled], ignore_index=True)
df_annulled.to_csv(annulled_path, index=False, encoding='utf-8-sig')
print(f'{annulled_path}: {len(df_annulled)} annulled questions from {df_annulled["exam"].nunique()} exams')